# StockSense — Predicting Daily Stock Movement with Machine Learning

**Goal:** Build a complete ML pipeline that predicts whether a stock will close *higher or lower* the next trading day.

**Why this problem?**
I chose this problem because stock price movement is one of the most studied, most difficult prediction tasks in ML. It's noisy, non-stationary, and full of many, common pitfalls with data. This makes it an excellent project for learning: every concept I studied here — data leakage, feature engineering, class imbalance, overfitting — is something engineers genuinely wrestle with in production systems.

**End deliverable:** A single, fully documented jupyter notebook — raw data in, trained models and evaluation charts out.

## 0. Imports

All dependencies are declared at the top of the notebook in a single cell. This is standard practice in data science notebooks. Full requirements can be found in the requirments.txt file in the project root directory.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as npy
import matplotlib.pyplot as plt
import seaborn as sbn

## 1.1 — Acquiring the Data

**What we're doing:** Downloading 5 years of daily OHLCV data for Apple Inc. (AAPL) from Yahoo Finance via the `yfinance` API.

**Why Apple (AAPL)?**
- Highly liquid — prices reflect genuine market activity
- Has 5 years of history that gives ~1,260 trading days, which is enough for meaningful ML without being unwieldy
- No survivorship bias concern for a single ticker used as a learning exercise
- Free and accessible via `yfinance`

**OHLCV defined:**
| Column | Meaning |
|--------|---------|
| Open | Price at market open |
| High | Intraday highest price |
| Low | Intraday lowest price |
| Close | Price at market close |
| Volume | Number of shares traded |
| Adj Close | Close adjusted for dividends and stock splits |

> 🔑 **Decision — raw Close vs Adj Close:**
> We are using `Adj Close` for feature calculations rather than `Close`. When a company pays a dividend or does a stock split, the raw Close price jumps discontinuously — not because of market sentiment, but because of a mechanical accounting event. `Adj Close` retroactively adjusts all historical prices to remove these jumps, giving a clean continuous series. Using raw `Close` here would introduce artificial signals into our features.

In [ ]:
df = yf.download('AAPL', period='5y')

### Inspecting the data — never skip this

Before any transformation, we run three standard inspection calls. This is called a **sanity check**, and skipping it is one of the most common mistakes junior engineers make. You will waste hours debugging a model that has a broken input, not a broken architecture.

- `df.head()` — confirm the column names, index type (should be `DatetimeIndex`), and that values look reasonable
- `df.info()` — check dtypes and the count of non-null entries per column. Any column with missing values will show a lower count than the total rows
- `df.describe()` — check the statistical summary: min, max, mean, std. A negative minimum Volume, for example, would immediately flag a data issue

> 🔑 **What we're watching for:**
> - Missing values (`NaN`s) — yfinance data is generally clean, but market holidays and early data can introduce gaps
> - Unexpected dtypes — all price/volume columns should be `float64`
> - Extreme outliers — a single row where High < Low would indicate a data error

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
data_size = len(df)
train_df = df.iloc[:int(data_size * 0.7)]
val_df = df.iloc[int(data_size * 0.7):int(data_size * 0.85)]
test_df = df.iloc[int(data_size * 0.85):]

In [ ]:
print(f'Train set shape: {train_df.shape}')
print(f'Val set shape:   {val_df.shape}')
print(f'Test set shape:  {test_df.shape}')
if train_df.shape[0] + val_df.shape[0] + test_df.shape[0] == data_size:
    print('Data split correctly!')
else:
    print('Data was not split correctly!')

In [ ]:
df['SMA10'] = df['Close'].rolling(10).mean()
df['SMA50'] = df['Close'].rolling(50).mean()
df[['Close', 'SMA10', 'SMA50']].head(50)